# Hierarchical Clustering — from scratch

Hierarchical clustering builds a **tree of nested clusters** (a *dendrogram*).

* **Agglomerative (bottom-up)**: start with every point as its own cluster; repeatedly merge the two closest clusters until one remains.
* **Divisive (top-down)**: start with one cluster and recursively split it (e.g. with 2-means — "bisecting K-Means").

Cutting the dendrogram at some height gives a flat clustering — no need to choose $k$ before running.

### Linkage: distance between clusters $A$ and $B$
| Linkage | $d(A,B)$ | Behaviour |
|---|---|---|
| Single | $\min_{a\in A,b\in B} d(a,b)$ | Long "chained" clusters, handles non-convex shapes |
| Complete | $\max_{a,b} d(a,b)$ | Compact clusters of similar diameter |
| Average (UPGMA) | mean of all pairwise distances | Compromise |
| Ward | increase in total within-cluster variance | Compact, similar-sized clusters (like K-Means) |

### Lance–Williams update
After merging $i$ and $j$, the distance from the new cluster to any other cluster $k$ can be computed from old distances:
$$d(i\cup j, k) = \alpha_i d(i,k) + \alpha_j d(j,k) + \beta d(i,j) + \gamma |d(i,k) - d(j,k)|$$

| Linkage | $\alpha_i$ | $\beta$ | $\gamma$ |
|---|---|---|---|
| Single | 1/2 | 0 | −1/2 |
| Complete | 1/2 | 0 | +1/2 |
| Average | $\frac{n_i}{n_i+n_j}$ | 0 | 0 |
| Ward (on squared distances) | $\frac{n_i+n_k}{n_i+n_j+n_k}$ | $\frac{-n_k}{n_i+n_j+n_k}$ | 0 |

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

## Agglomerative clustering

In [ ]:
class AgglomerativeClustering:
    """Naive O(n^3) agglomerative clustering with Lance-Williams updates.

    Produces a linkage matrix Z (same format as scipy): row t = [cluster_a, cluster_b, distance, size];
    original points are clusters 0..n-1 and the cluster created at step t gets id n + t.
    """

    def __init__(self, linkage="ward"):
        assert linkage in ("single", "complete", "average", "ward")
        self.linkage = linkage

    def fit(self, X):
        n = len(X)
        D = np.sqrt(np.maximum((X ** 2).sum(1)[:, None] + (X ** 2).sum(1)[None, :] - 2 * X @ X.T, 0))
        if self.linkage == "ward":
            D = D ** 2                                     # Ward's recurrence works on squared distances
        np.fill_diagonal(D, np.inf)

        active = np.ones(n, dtype=bool)
        size = np.ones(n)
        cluster_id = np.arange(n)                          # which cluster each row of D currently represents
        Z = np.zeros((n - 1, 4))

        for step in range(n - 1):
            i, j = np.unravel_index(np.argmin(D), D.shape)
            i, j = min(i, j), max(i, j)
            d_ij = D[i, j]
            Z[step] = [cluster_id[i], cluster_id[j], np.sqrt(d_ij) if self.linkage == "ward" else d_ij,
                       size[i] + size[j]]

            others = active.copy(); others[[i, j]] = False
            di, dj = D[i, others], D[j, others]
            ni, nj, nk = size[i], size[j], size[others]
            if self.linkage == "single":
                new = np.minimum(di, dj)                   # = 0.5 di + 0.5 dj - 0.5 |di - dj|
            elif self.linkage == "complete":
                new = np.maximum(di, dj)                   # = 0.5 di + 0.5 dj + 0.5 |di - dj|
            elif self.linkage == "average":
                new = (ni * di + nj * dj) / (ni + nj)
            else:
                new = ((ni + nk) * di + (nj + nk) * dj - nk * d_ij) / (ni + nj + nk)

            # row i now represents the merged cluster; row j is retired
            D[i, others] = D[others, i] = new
            D[j, :] = D[:, j] = np.inf
            active[j] = False
            size[i] += size[j]
            cluster_id[i] = n + step

        self.Z = Z
        self.n_samples = n
        return self

    def labels(self, n_clusters):
        """Cut the tree: replay the first n - n_clusters merges with union-find."""
        n = self.n_samples
        parent = list(range(2 * n - 1))

        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        for t in range(n - n_clusters):
            a, b = int(self.Z[t, 0]), int(self.Z[t, 1])
            parent[find(a)] = n + t
            parent[find(b)] = n + t
        roots = [find(i) for i in range(n)]
        _, labels = np.unique(roots, return_inverse=True)
        return labels

## Drawing a dendrogram from the linkage matrix

In [ ]:
def plot_dendrogram(Z, ax=None, cut_height=None, title="Dendrogram"):
    ax = ax or plt.gca()
    n = len(Z) + 1
    children = {n + t: (int(Z[t, 0]), int(Z[t, 1])) for t in range(len(Z))}

    # left-to-right leaf order via an explicit stack (avoids deep recursion)
    order, stack = [], [2 * n - 2]
    while stack:
        node = stack.pop()
        if node < n:
            order.append(node)
        else:
            a, b = children[node]
            stack.extend([b, a])

    x = {leaf: i for i, leaf in enumerate(order)}
    h = {leaf: 0.0 for leaf in order}
    for t, (a, b, dist, _) in enumerate(Z):
        a, b = int(a), int(b)
        ax.plot([x[a], x[a], x[b], x[b]], [h[a], dist, dist, h[b]], c="tab:blue", lw=1)
        x[n + t], h[n + t] = (x[a] + x[b]) / 2, dist
    if cut_height is not None:
        ax.axhline(cut_height, c="red", ls="--", label="cut")
        ax.legend()
    ax.set_xticks([] if n > 40 else range(n))
    if n <= 40:
        ax.set_xticklabels(order, fontsize=8)
    ax.set_ylabel("merge distance"); ax.set_title(title)

## A small example

In [ ]:
X_small, y_small = make_blobs(n_samples=24, centers=3, cluster_std=0.8, random_state=4)
agg = AgglomerativeClustering("average").fit(X_small)
print("First 5 merges [a, b, distance, size]:\n", np.round(agg.Z[:5], 3))

labels = agg.labels(3)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(X_small[:, 0], X_small[:, 1], c=labels, cmap="viridis", s=60)
for i, (px, py) in enumerate(X_small):
    axes[0].annotate(str(i), (px, py), fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[0].set_title("Points (coloured by 3-cluster cut)")
cut = (agg.Z[-3, 2] + agg.Z[-2, 2]) / 2                  # any height between the last 3 merges gives 3 clusters
plot_dendrogram(agg.Z, axes[1], cut_height=cut, title="Average-linkage dendrogram")
plt.tight_layout(); plt.show()

## Comparing linkage criteria

In [ ]:
Xb, yb = make_blobs(n_samples=150, centers=3, cluster_std=[1.0, 2.0, 0.5], random_state=8)
Xm, ym = make_moons(n_samples=150, noise=0.05, random_state=0)

linkages = ["single", "complete", "average", "ward"]
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
for col, link in enumerate(linkages):
    agg_b = AgglomerativeClustering(link).fit(Xb)
    agg_m = AgglomerativeClustering(link).fit(Xm)
    axes[0, col].scatter(Xb[:, 0], Xb[:, 1], c=agg_b.labels(3), cmap="viridis", s=12)
    axes[0, col].set_title(f"{link} linkage — blobs")
    axes[1, col].scatter(Xm[:, 0], Xm[:, 1], c=agg_m.labels(2), cmap="viridis", s=12)
    axes[1, col].set_title(f"{link} linkage — moons")
    plot_dendrogram(agg_b.Z, axes[2, col], title=f"{link} dendrogram (blobs)")
plt.tight_layout(); plt.show()

## Divisive clustering: bisecting K-Means
Top-down alternative: repeatedly split the cluster with the largest SSE into two using 2-means.

In [ ]:
def two_means(X, n_iter=30, seed=0):
    rng = np.random.RandomState(seed)
    centers = X[rng.choice(len(X), 2, replace=False)]
    for _ in range(n_iter):
        labels = np.argmin(((X[:, None] - centers[None]) ** 2).sum(-1), axis=1)
        centers = np.array([X[labels == j].mean(0) if np.any(labels == j) else centers[j] for j in range(2)])
    return labels


def bisecting_kmeans(X, n_clusters):
    clusters = [np.arange(len(X))]
    while len(clusters) < n_clusters:
        sse = [((X[c] - X[c].mean(0)) ** 2).sum() for c in clusters]
        target = clusters.pop(int(np.argmax(sse)))
        split = two_means(X[target])
        clusters += [target[split == 0], target[split == 1]]
    labels = np.empty(len(X), dtype=int)
    for k, idx in enumerate(clusters):
        labels[idx] = k
    return labels


plt.scatter(Xb[:, 0], Xb[:, 1], c=bisecting_kmeans(Xb, 3), cmap="viridis", s=15)
plt.title("Divisive (bisecting K-Means), 3 clusters"); plt.show()

## Summary
* Produces a full hierarchy; choose the number of clusters afterwards by cutting the dendrogram (large vertical gaps suggest natural cuts).
* Linkage choice changes results dramatically: **single** follows chains (good for moons, bad with noise), **Ward/complete** give compact clusters.
* Naive agglomerative clustering costs $O(n^2)$ memory and $O(n^3)$ time — fine for thousands of points, not millions.